In [1]:
#| default_exp frida

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'fred'

In [6]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration 
tokenizer = GPT2Tokenizer.from_pretrained(full_path, eos_token='</s>')
model = T5ForConditionalGeneration.from_pretrained(full_path, torch_dtype=torch.bfloat16) 
device='cuda'
model.to(device);
model.eval();


In [7]:
model

T5ForConditionalGeneration(
  (shared): Embedding(50364, 1536)
  (encoder): T5Stack(
    (embed_tokens): Embedding(50364, 1536)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1536, out_features=1536, bias=False)
              (k): Linear(in_features=1536, out_features=1536, bias=False)
              (v): Linear(in_features=1536, out_features=1536, bias=False)
              (o): Linear(in_features=1536, out_features=1536, bias=False)
              (relative_attention_bias): Embedding(32, 24)
            )
            (layer_norm): FusedRMSNorm(torch.Size([1536]), eps=1e-06, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1536, out_features=4096, bias=False)
              (wi_1): Linear(i

In [8]:
sum(p.numel() for p in model.parameters())

1740354048

In [53]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    output_ids = model.generate(input_ids, do_sample=True, temperature=.5, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, 
                        num_return_sequences=num_samples,)

    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [18]:
# #| export
# def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
#    return generate(model, tokenizer, seq_length, '<LM>' + prompt, length, num_samples, allow_linebreak)

In [19]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 2.07 s, sys: 18.5 ms, total: 2.09 s
Wall time: 2.06 s


['за из «Грачи улетели», где «не стреляйте в пианиста, он играет, как умеет», стала крылатой?',
 'артман, чтоб избежать лекций по юриспруденции и совместить все свои вожделения. Но я знаю, что твоя принадлежность к русской интеллигенции не имеет никакого отношения к твоему литературному творчеству.',
 'столь же глубокомысленную паузу. И у меня сразу возникают нехорошие предчувствия. Он знает, подумал я. Чего же он ждет? Может, вам стоит поговорить?',
 ' Камаса». Как сказал бы Омар Хайям, «отданная душа потеряна в веках». Будь Омаром Хайямом Стрингер, он легко заработал бы себе прозвище «легендарный разбойник».']

In [54]:
%%time
get_sample_casual('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.49 s, sys: 0 ns, total: 1.49 s
Wall time: 1.48 s


[' – говно. –\xa0 Я не Лев Толстой, а Николай Второй… И я уже давно ни на что и никому по большому счету плевать хотел! А ты все равно мне завидуешь?',
 ' — говно». —\xa0 А что, у вас есть другие варианты? спросил я.',
 ' – просто говно. –\xa0 А ты, значит… Я думал о тебе лучше! Ты не такая уж и сволочь!» И я стал думать дальше: «А что если это все-таки правда? Что тогда?',
 ' – просто говно. –\xa0 Почему? Я же не говорю, что я Лев Толстой… Хотя ты прав: на словах у меня есть и то правда тоже».']

tensor([    0,   203,    12,   369,    13,   225,   385,    18, 32384,  4205,
          478, 21587,  1711,   290,  8401, 21198,   861,     2],
       device='cuda:0')

In [47]:
lm_text = '<LM>На словах ты Лев Толстой, а на деле' 
input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
output_ids = model.generate(input_ids, do_sample=True, temperature=.5, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                    max_new_tokens=4000, )

result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
result = process_seq(result)

print(result)



[' – говно. –\xa0 А что такое « на словах»? спросил я, но тут же понял: это не вопрос и даже вовсе необязательно отвечать… В любом случае ответ был очевиден — в его глазах была насмешка над самим собой; он как бы говорил мне ( или себе): „ Да ладно тебе!“ Но вот чего- то другого во взгляде у него все равно было больше всего— какой‑то странной теплоты при воспоминании о чем− нибудь хорошем из прошлого... И еще эта странная улыбка сквозь слезы,— словно ему вспоминалось нечто очень хорошее с ним вместе когда\x1e либо прошедшее по жизни.— Я вдруг почувствовал себя ужасно старым». Это воспоминание почему-- та особенно сильно взволновало меня тогда (« Может быть потому», подумал про нее Лев Толстой), хотя оно никак нельзя сказать чтобы имело к нему какое‐ нить отношение вообще.. Скорее наоборот - именно оттого она показалась такой важной для понимания происходящего со мной сейчас!.. Впрочем нет ― ничего такого особенного там просто напросто уже давно никто никогда ни от кого особо близко ожи

In [48]:
result

[' – говно. –\xa0 А что такое «на словах»? спросил я, но тут же понял: это не вопрос и даже вовсе необязательно отвечать… В любом случае ответ был очевиден — в его глазах была насмешка над самим собой; он как бы говорил мне (или себе): „Да ладно тебе!“ Но вот чего-то другого во взгляде у него все равно было больше всего— какой‑то странной теплоты при воспоминании о чем− нибудь хорошем из прошлого... И еще эта странная улыбка сквозь слезы,— словно ему вспоминалось нечто очень хорошее с ним вместе когда\x1e либо прошедшее по жизни.— Я вдруг почувствовал себя ужасно старым». Это воспоминание почему-- та особенно сильно взволновало меня тогда (« Может быть потому», подумал про нее Лев Толстой), хотя оно никак нельзя сказать чтобы имело к нему какое‐ нить отношение вообще.. Скорее наоборот - именно оттого она показалась такой важной для понимания происходящего со мной сейчас!.. Впрочем нет ― ничего такого особенного там просто напросто уже давно никто никогда ни от кого особо близко ожидать